# 1장 클린 아키텍처의 핵심: 파이썬 개발의 변화

파이썬으로 구현하는 클린 아키텍처 - 1장 클린 아키텍처의 핵심: 파이썬 개발의 변화 코드 예제

## 개요

이 장에서는 클린 아키텍처가 무엇인지, 왜 중요한지 ﻿살펴본다.

이 장에서 다루는 주요 주제:
* 파이썬에서 클린 아키텍처가 필요한 이유: 계획과 개발 민첩성의 조화가 주는 장점
* 클린 아키텍처란 무엇인가?
* 클린 아키텍처와 파이썬: 자연스러운 조합

### 00_notification_service_ABC.py

## ABC(추상 기본 클래스)를 사용한 알림 서비스

클린 아키텍처의 핵심 원칙은 구체적인 구현보다 추상화에 의존하는 것이다. 이 원칙은 앞서 논의한 의존성 규칙, 즉 의존성은 반드시 안쪽으로만 향해야 한다는 규칙을 직접적으로 뒷받침한다.

이 예제는 파이썬의 ABC를 사용해 클린 아키텍처의 핵심 개념을 보여 준다.
* ABC: Notifier 클래스는 ABC로, 모든 알림 클래스가 따라야 할 인터페이스를 정의한다. 클린 아키텍처에서 내부 원에 해당한다.
* 추상 메서드: Notifier의 send_notification 메서드는 @abstractmethod로 표시되어 하위 클래스에서 반드시 구현해야 한다.
* 구체 클래스: EmailNotifier와 SMSNotifier는 외부 원에 있는 구체 클래스다. Notifier를 상속받아 실제로 알림을 어떻게 보낼지 구현한다.
* 의존성 역전: NotificationService 클래스는 구체 클래스가 아닌 추상 Notifier 클래스에 의존한다. 추상 Notifier(내부 원)가 구체 알림 클래스들(외부 원)에 의존하지 않으므로 의존성 규칙을 지킬 수 있다. 의존성 역전은 다음 장에서 자세히 다룬다.
이 구조를 통해 앞서 살펴본 클린 아키텍처 원칙을 구현한다.

In [1]:
# ABC(추상 기본 클래스)를 활용한 알림 서비스 예제
# 클린 아키텍처의 핵심 원칙인 "추상화에 의존하라"를 보여주는 코드
from abc import ABC, abstractmethod


# 알림 발송의 추상 인터페이스 (클린 아키텍처의 내부 원에 해당)
# 모든 알림 클래스가 반드시 구현해야 할 계약(contract) 정의
class Notifier(ABC):
    @abstractmethod
    def send_notification(self, message: str) -> None:
        # 하위 클래스에서 반드시 구현해야 하는 추상 메서드
        pass


# 이메일 방식의 구체적 알림 구현체 (외부 원에 해당)
# Notifier 추상 클래스를 상속받아 실제 이메일 발송 로직 구현
class EmailNotifier(Notifier):
    def send_notification(self, message: str) -> None:

        print(f"이메일 발송: {message}")


# SMS 방식의 구체적 알림 구현체 (외부 원에 해당)
# Notifier 추상 클래스를 상속받아 실제 SMS 발송 로직 구현
class SMSNotifier(Notifier):
    def send_notification(self, message: str) -> None:

        print(f"SMS 발송: {message}")


# 알림 서비스 클래스 - 의존성 역전 원칙(DIP)의 적용
# 구체 클래스(EmailNotifier, SMSNotifier)가 아닌 추상 클래스(Notifier)에 의존
# → 새로운 알림 방식 추가 시 이 클래스를 수정할 필요 없음 (개방-폐쇄 원칙)
class NotificationService:
    def __init__(self, notifier: Notifier):
        # 생성자를 통한 의존성 주입 - 외부에서 구체적 알림 구현체를 전달
        self.notifier = notifier

    def notify(self, message: str) -> None:
        # 추상 인터페이스의 메서드 호출 - 실제 구현체가 무엇인지 알 필요 없음
        self.notifier.send_notification(message)


# 사용 예시
# EmailNotifier 구체 클래스 생성 후 NotificationService에 주입
email_notifier = EmailNotifier()
email_service = NotificationService(email_notifier)
email_service.notify("이메일을 통한 인사")

이메일 발송: 이메일을 통한 인사


### 01_notification_service_protocol.py

## Protocol을 사용한 구조적 타이핑(Structural Typing)

간단한 ABC 예제를 살펴봤는데, 바로 여기서 파이썬이 진정으로 빛을 발한다. 클래스 계층 구조를 사용하지 않고도 동일한 클린 아키텍처 원칙을 구현할 수 있으며, 대신 파이썬의 덕 타이핑duck typing을 활용할 수 있다.

In [2]:
# Protocol을 사용한 구조적 타이핑(Structural Typing) 예제
# ABC와 달리 명시적 상속 없이 덕 타이핑으로 인터페이스를 구현하는 파이썬다운 방식
from typing import Protocol

# Protocol 기반 알림 인터페이스 정의
# ABC와 달리 상속 강제 없이, 동일한 메서드 시그니처만 있으면 호환되는 구조적 타이핑
class Notifier(Protocol):
    def send_notification(self, message: str) -> None:
        ...

# 명시적 상속 없이 Notifier 프로토콜을 암묵적으로 구현하는 이메일 알림 클래스
# send_notification 메서드 시그니처가 일치하므로 자동으로 Notifier로 인식
class EmailNotifier:  # 참고: 명시적 상속 없음
    def send_notification(self, message: str) -> None:

        print(f"이메일 발송 중: {message}")

# 명시적 상속 없이 Notifier 프로토콜을 암묵적으로 구현하는 SMS 알림 클래스
class SMSNotifier:  # 참고: 명시적 상속 없음
    def send_notification(self, message: str) -> None:

        print(f"SMS 발송 중: {message}")

# 알림 서비스 - Protocol 타입 힌트를 통한 느슨한 결합
# ABC 방식과 동일한 의존성 역전을 상속 없이 달성
class NotificationService:
    def __init__(self, notifier: Notifier):  # 여전히 타입 힌팅 사용 가능
        self.notifier = notifier

    def notify(self, message: str) -> None:
        self.notifier.send_notification(message)

# 사용법 - SMSNotifier가 Notifier를 상속하지 않았지만 프로토콜 호환으로 사용 가능
sms_notifier = SMSNotifier()
sms_service = NotificationService(sms_notifier)
sms_service.notify("안녕하세요?")

SMS 발송 중: 안녕하세요?
